# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [9]:
import os

path = 'D:/Proyectos/RI/01basicsearch/data/01_corpus_turismo_500.txt'

# Leer el archivo línea por línea
with open(path, 'r', encoding='utf-8') as f:
    files = f.readlines()

In [10]:
# Eliminar líneas vacías y espacios al inicio/final
documentos_turismo = [line.strip() for line in files if line.strip()]

print(f"Número de documentos en el corpus de turismo: {len(documentos_turismo)}")

Número de documentos en el corpus de turismo: 500


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Creamos la matriz TF-IDF
matriz_turismo = TfidfVectorizer()

# Calcular la matriz TF-IDF
tfidf_matrix_turismo = matriz_turismo.fit_transform(documentos_turismo)

# Mostrar dimensiones de la matriz (documentos x términos)
print(f"Dimensión de la matriz TF-IDF: {tfidf_matrix_turismo.shape}")
print("Ejemplo de términos en el vocabulario:", list(matriz_turismo.vocabulary_.keys())[:10])

Dimensión de la matriz TF-IDF: (500, 116)
Ejemplo de términos en el vocabulario: ['otavalo', 'es', 'conocido', 'por', 'su', 'mercado', 'indígena', 'artesanía', 'perfecto', 'para']


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [16]:
gutenberg_dir = "D:/Proyectos/RI/01basicsearch/data/gutenberg_1000/"

def cargar_corpus_gutenberg(ruta_carpeta):
    """
    Lee todos los archivos .txt de la carpeta y devuelve:
        - documentos: lista de strings (contenido de cada libro)
        - nombres: lista con los nombres de los archivos
    """
    documentos = []
    nombres = []
    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".txt"):
            ruta_completa = os.path.join(ruta_carpeta, archivo)
            with open(ruta_completa, 'r', encoding='utf-8') as f:
                contenido = f.read()
            documentos.append(contenido)
            nombres.append(archivo)
    return documentos, nombres

In [17]:
documentos_gutenberg, nombres_gutenberg = cargar_corpus_gutenberg(gutenberg_dir)
print(f"Se cargaron {len(documentos_gutenberg)} documentos.")

Se cargaron 1000 documentos.


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [18]:
vectorizer_gutenberg = TfidfVectorizer()

# Calcular la matriz TF-IDF (documentos x términos)
tfidf_matrix_gutenberg = vectorizer_gutenberg.fit_transform(documentos_gutenberg)

# Mostrar dimensiones
print(f"Matriz TF-IDF para Gutenberg 1000: {tfidf_matrix_gutenberg.shape}")
print("Tamaño del vocabulario:", len(vectorizer_gutenberg.vocabulary_))

Matriz TF-IDF para Gutenberg 1000: (1000, 1101409)
Tamaño del vocabulario: 1101409


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [19]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def buscar(query, top_k=10):
    """
    Realiza una búsqueda sobre el corpus Gutenberg 1000 usando el modelo vectorial.
    
    Parámetros:
        query (str): Consulta del usuario.
        top_k (int): Número de resultados a retornar.
    
    Retorna:
        list: Lista de tuplas (nombre_archivo, similitud_coseno) ordenadas de mayor a menor.
        También imprime el tiempo de búsqueda.
    """
    start_time = time.time()  # Inicio del temporizador
    
    # Transformar la consulta a vector TF-IDF usando el mismo vectorizador del corpus
    query_vector = vectorizer_gutenberg.transform([query])
    
    # Calcular similitud coseno entre la consulta y todos los documentos
    sim_scores = cosine_similarity(query_vector, tfidf_matrix_gutenberg).flatten()
    
    # Obtener los índices de los top_k documentos más similares
    top_indices = sim_scores.argsort()[-top_k:][::-1]
    
    # Construir lista de resultados
    resultados = []
    for idx in top_indices:
        if sim_scores[idx] > 0:   # Solo mostrar documentos con similitud > 0
            resultados.append((nombres_gutenberg[idx], sim_scores[idx]))
    
    end_time = time.time()         # Fin del temporizador
    elapsed_time = end_time - start_time
    print(f"Tiempo de búsqueda: {elapsed_time:.4f} segundos")
    
    # Mostrar resultados (opcional)
    print(f"\nTop {len(resultados)} resultados para la consulta: '{query}'\n")
    for i, (doc, score) in enumerate(resultados, 1):
        print(f"{i}. {doc} (similitud = {score:.4f})")
    
    return resultados

In [21]:
# Prueba de la función con una consulta de ejemplo
ejemplo = "Dracula"
resultados_ejemplo = buscar(ejemplo, top_k=5)

Tiempo de búsqueda: 0.7509 segundos

Top 1 resultados para la consulta: 'Dracula'

1. book_9109.txt (similitud = 0.0008)
